# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**The decision I feed:** which pages should a reviewer open *first*. A reviewer opens K pages and K is the budget (20 / 50 / 100), so this is a **ranking** job, not an accuracy job. The honest metric is **precision@K**: of the top K pages, how many really declined, measured on clients the model never met.

**Why these methods, in this order:**
- **Logistic Regression first** — it is readable. It prints one weight per feature, so I can see what the model leans on and argue with it before believing it. It fits a yes/no (declined or not) by bending a weighted sum into a probability between 0 and 1.
- **Shallow Decision Tree next** — a depth-3 tree I can read out loud; it shows the logic. Its weakness (few distinct scores, so ties at the K cut) is exactly what the tie policy exists to make honest.
- **Random Forest last** — decline patterns here are likely tangled (a big old page does not behave like a big new page), and the forest lets features interact without me hand-crafting every interaction. Many small trees each memorise something different, and the mistakes cancel out.
- **No gradient boosting** — the forest is already enough complexity to test the pattern. Boosting would add a tuning surface on a small, lopsided fold portfolio without a cleaner question to answer. Complexity has to earn its place on the same test.

**What I refuse to claim:** neither method *causes* decline. They rank pages by the measured signal that lives before the March label window. The frozen Week-4 rule is the bar to beat; the base rate is the floor under the floor.

In [1]:
# Section 1 — rebuild the SAME frame the frozen Week-4 baseline was scored on,
# then define the model features and the label. No new data, no new label.

%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass, duckdb, hashlib, json
import pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Feature window Jan 30 - Feb 28 (closed before the label window). Label window March:
# imp_last30 < 0.8 * imp_prev30. Same eligibility as the frozen baseline: gsc available, imp_prev30 >= 100.
df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END) AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)

# Feature engineering: the five fields the rule used, with the two heavy-tailed click/impression
# counts log-scaled so a linear model is not run by one giant page. pos=0 / age=0 mean 'no data',
# so missing values get 0 (documented, not a blind fill).
n_missing = int(df[['pos_prev30', 'content_age_days']].isna().any(axis=1).sum())
df['pos_prev30'] = df['pos_prev30'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)
df['log_imp_prev30'] = np.log1p(df['imp_prev30'])
df['log_clk_prev30'] = np.log1p(df['clk_prev30'])

FEATURES = ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']

print(f'Frame: {len(df):,} pages, {df["client_hash_id"].nunique()} clients')
print(f'Base rate (whole frame): {df["is_declining"].mean():.3f}')
print(f'Missing pos/age rows filled as 0 (no-data): {n_missing}')
print(f'Model features (all knowable before March 1): {FEATURES}')
print(f'Label-window field (imp_last30) in features? False')


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Frame: 81,521 pages, 37 clients
Base rate (whole frame): 0.249
Missing pos/age rows filled as 0 (no-data): 0
Model features (all knowable before March 1): ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
Label-window field (imp_last30) in features? False


## 2. Split design

**Grouped by client — five deterministic hash folds, every client on ONE side.** This is the honest split for the deployment question: *"would this work for a client we have not met yet?"* A random split would let the same clients sit on both sides, so the model would be graded on pages from clients it already knows — free marks. Grouping keeps the judged pages from clients the model never trained on, so a good score has to come from the pattern, not from the name.

**Same contract as the frozen Week-4 baseline:** same 5 folds, same test rows, same K (20/50/100), same metric (precision@K), same tie policy (score desc, then seeded content-hash asc). The split is a pure hash function, so re-running either notebook reproduces the same folds. The base rate is printed per fold because the folds start from very different decline rates — some queues are simply easier than others.

In [2]:
# Section 2 — the grouped-by-client split, with the overlap assertion the contract requires
import hashlib

def client_fold(client_id, n_folds=5):
    return int(hashlib.sha256(client_id.encode()).hexdigest(), 16) % n_folds

df['fold'] = df['client_hash_id'].map(client_fold)
df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))

# Assertions: no content item lands on both sides; train/test clients are disjoint per fold.
assert df.groupby('content_hash_id')['fold'].nunique().max() == 1, 'a content item crosses folds'
print('Split check: every content item sits in exactly one fold (pass)')
print()
print(df.groupby('fold').size().rename('pages').to_string())
print(df.groupby('fold')['client_hash_id'].nunique().rename('clients').to_string())
print()
for f in sorted(df['fold'].unique()):
    tr = df[df['fold'] != f]; te = df[df['fold'] == f]
    assert not (set(tr['client_hash_id']) & set(te['client_hash_id'])), f'fold {f}: clients overlap'
    print(f'fold {int(f)}: train {len(tr):>7,} pages / {tr["client_hash_id"].nunique():>3} clients | '
          f'test {len(te):>7,} pages / {te["client_hash_id"].nunique():>3} clients | '
          f'test base rate {te["is_declining"].mean():.3f}')
print()
print('Contract: same folds, same rows, same K, same tie policy as the Week-4 baseline.')

Split check: every content item sits in exactly one fold (pass)

fold
0    25520
1     4276
2     6675
3    16084
4    28966
fold
0    9
1    8
2    6
3    5
4    9

fold 0: train  56,001 pages /  28 clients | test  25,520 pages /   9 clients | test base rate 0.215


fold 1: train  77,245 pages /  29 clients | test   4,276 pages /   8 clients | test base rate 0.128
fold 2: train  74,846 pages /  31 clients | test   6,675 pages /   6 clients | test base rate 0.665


fold 3: train  65,437 pages /  32 clients | test  16,084 pages /   5 clients | test base rate 0.274
fold 4: train  52,555 pages /  28 clients | test  28,966 pages /   9 clients | test base rate 0.186

Contract: same folds, same rows, same K, same tie policy as the Week-4 baseline.


## 3. Train + compare vs my frozen baseline

Same rows, same folds, same metric, same K, same tie policy as the Week-4 receipt. Inside every fold the model is fitted **only on the four training folds** and scored **on the fifth (unseen clients)**. The frozen rule baseline is recomputed here on the very same test rows, so the two numbers compete on the same exam. A fixed seed is used for every model, and **no hyperparameter search** is run against the held-out fold — that would turn the test fold into part of the model.

**Measured reading:** on unseen clients the forest leads on average — mean fold P@50 **0.67** vs the frozen rule's **0.38** and the shallow tree's **0.42** — but it is not a stable winner: it loses fold 4 (0.26 vs the rule's 0.34), and the folds swing from 0.26 to 1.00. Logistic Regression (mean 0.15) loses to the rule outright, so a straight linear view is too weak here — the tangled signal is exactly what the forest earns its place on. No winner is declared from the average alone; direction and wobble are reported together.

In [3]:
# Section 3 — fit on train folds only, score held-out clients, compare on the same rows
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

SEED = 42
K_VALUES = [20, 50, 100]
print(f'sklearn {sklearn.__version__} | seed {SEED}')
print()

# The frozen rule, recomputed on the same test rows (Week-4 contract, tie policy included).
def rule_score(row):
    stale = 1.0 if row['content_age_days'] >= 90 else 0.0
    visible = 1.0 if row['imp_prev30'] >= 500 else 0.0
    return stale * visible * row['imp_prev30']
df['baseline_score'] = df.apply(rule_score, axis=1)

def precision_at_k(sorted_labels, k):
    head = sorted_labels.head(min(k, len(sorted_labels)))
    return float(head.mean()) if len(head) else float('nan')

models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    'tree_d3': DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=SEED),
    'forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
}

X = df[FEATURES].to_numpy()
y = df['is_declining'].to_numpy()

rows = []
for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    test = df[te_mask].copy()
    rec = {'fold': int(f), 'n_test': int(len(test)), 'base_rate': round(float(test['is_declining'].mean()), 4)}

    # baseline on the SAME test rows
    te_base = test.sort_values(['baseline_score', '_tie'], ascending=[False, True]).reset_index(drop=True)
    for k in K_VALUES:
        rec[f'baseline_precision@{k}'] = round(precision_at_k(te_base['is_declining'], k), 4)

    # learned models: fit on train folds only, score the held-out fold
    for name, model in models.items():
        model.fit(X[tr_mask], y[tr_mask])
        proba = model.predict_proba(X[te_mask])[:, 1]
        te_mod = test.assign(score=proba).sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
        for k in K_VALUES:
            rec[f'{name}_precision@{k}'] = round(precision_at_k(te_mod['is_declining'], k), 4)
        rec[f'{name}_auc'] = round(float(roc_auc_score(test['is_declining'], proba)), 4)
        if len(te_mod) >= 50:
            rec[f'{name}_ties_at_50'] = int((te_mod['score'] == te_mod.loc[49, 'score']).sum())
    rows.append(rec)

summary = pd.DataFrame(rows)

print('=== Model vs frozen baseline — same test rows, same K, same tie policy ===')
cols = ['fold', 'n_test', 'base_rate']
for k in K_VALUES:
    cols += [f'baseline_precision@{k}', f'logistic_precision@{k}', f'tree_d3_precision@{k}', f'forest_precision@{k}']
print(summary[cols].to_string(index=False))
print()

print('=== Headline: precision@50 — every fold kept visible, base rate beside it ===')
hdr = ['base_rate', 'baseline_precision@50', 'logistic_precision@50', 'tree_d3_precision@50', 'forest_precision@50']
tbl = summary[['fold'] + hdr].set_index('fold')
print(tbl.to_string())
print()
for col in hdr:
    print(f'{col:26s} mean {tbl[col].mean():.4f}   min {tbl[col].min():.4f}   max {tbl[col].max():.4f}')
print()
for m in ['logistic', 'tree_d3', 'forest']:
    print(f'{m:26s} mean AUC {summary[f"{m}_auc"].mean():.4f}')
print()
print('=== Ties at the K=50 cut — rows sharing the cutoff score (tie-break must be defined) ===')
tie_cols = [c for c in summary.columns if c.endswith('ties_at_50')]
print(summary[['fold'] + tie_cols].to_string(index=False))
print()

# Cross-check: recomputed baseline must equal the frozen Week-4 receipt.
frozen = json.load(open(os.path.join(OUT_DIR, 'baseline_folds_receipt.json'), encoding='utf-8'))
frozen_p50 = {r['fold']: r['precision@50'] for r in frozen['folds']}
recomp_p50 = {int(r['fold']): r['baseline_precision@50'] for r in rows}
print('Recomputed baseline P@50 matches the frozen Week-4 receipt:', frozen_p50 == recomp_p50)
print()

receipt = {
    'seed': SEED,
    'split': 'client-grouped deterministic 5-way hash fold',
    'features': FEATURES,
    'metric': 'precision@K',
    'K': K_VALUES,
    'tie_policy': 'score desc, then seeded content-hash asc',
    'folds': rows,
    'mean_precision@50': {m: round(float(sum(r[f'{m}_precision@50'] for r in rows) / len(rows)), 4)
                          for m in ['baseline', 'logistic', 'tree_d3', 'forest']},
}
rec_path = os.path.join(OUT_DIR, 'model_vs_baseline_folds.json')
with open(rec_path, 'w') as fh:
    json.dump(receipt, fh, indent=2)
print(f'Receipt: {rec_path}')

sklearn 1.8.0 | seed 42



=== Model vs frozen baseline — same test rows, same K, same tie policy ===
 fold  n_test  base_rate  baseline_precision@20  logistic_precision@20  tree_d3_precision@20  forest_precision@20  baseline_precision@50  logistic_precision@50  tree_d3_precision@50  forest_precision@50  baseline_precision@100  logistic_precision@100  tree_d3_precision@100  forest_precision@100
    0   25520     0.2153                   0.10                   0.05                  0.20                 0.50                   0.20                   0.04                  0.32                 0.50                    0.18                    0.09                   0.24                  0.49
    1    4276     0.1284                   0.05                   0.00                  0.20                 0.95                   0.08                   0.04                  0.22                 0.98                    0.05                    0.05                   0.20                  0.91
    2    6675     0.6649             

## 4. Errors and interpretation

A metric says *how often* the model is wrong; the wrong rows say *why*. Three looks: what the forest actually leans on (permutation importance, with the spread across folds — not just the order), concrete wrong cases (a confident over-flag and a confident miss), and a sensitivity check that removes the click-count fields which sit close to the label definition.

**What the errors look like:** the forest is confident-wrong on pages that sat high in the results all February but earned almost no clicks (zero-click impressions are fragile — the model reads them as decay), and it is confidently safe-wrong on brand-new pages with only a handful of days of history that then dropped — thin history is where momentum-style signals mislead. The permutation-importance spread is wide: which feature leads depends on which clients were held out, so the lean is real but not stable. When the click/impression counts (close cousins of the label) are removed on a like-for-like forest, the lead shrinks only a little — the pattern is not carried by label leakage alone.

In [4]:
# Section 4 — errors, what the model leans on, and the label-cousin sensitivity
from sklearn.inspection import permutation_importance

# --- 1) Forest test predictions for every row (the wrong-case hunt) ---
pred = []
for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf.fit(X[tr_mask], y[tr_mask])
    tmp = df[te_mask].copy()
    tmp['forest_proba'] = rf.predict_proba(X[te_mask])[:, 1]
    pred.append(tmp)
pred = pd.concat(pred)

# --- 2) Permutation importance of the forest (mean + spread over folds) ---
importances = {feat: [] for feat in FEATURES}
for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf.fit(X[tr_mask], y[tr_mask])
    pi = permutation_importance(rf, X[te_mask], y[te_mask], n_repeats=4, random_state=SEED, n_jobs=-1)
    for feat, imp in zip(FEATURES, pi.importances_mean):
        importances[feat].append(float(imp))

imp_df = pd.DataFrame(importances).T
imp_df.columns = [f'fold{i}' for i in sorted(df['fold'].unique())]
imp_df['mean'] = imp_df.mean(axis=1)
imp_df['min'] = imp_df.min(axis=1)
imp_df['max'] = imp_df.max(axis=1)
print('Permutation importance — forest (higher = leaned harder; spread = how stable the lean is):')
print(imp_df.sort_values('mean', ascending=False).round(4).to_string())
print()

# --- 3) Wrong cases: over-flag (high proba but did NOT decline) and missed (declined but low proba) ---
flagged = pred[(pred['forest_proba'] >= 0.7) & (pred['is_declining'] == 0)].sort_values('forest_proba', ascending=False)
missed = pred[(pred['forest_proba'] <= 0.3) & (pred['is_declining'] == 1)].sort_values('forest_proba', ascending=True)
print(f'Over-flag (proba>=0.7, held steady): {len(flagged):,}  |  missed (proba<=0.3, declined): {len(missed):,}')
print()

def show_cases(rows_df, title, n=3):
    print(f'--- {title} ---')
    for _, r in rows_df.head(n).iterrows():
        print(f'client {r["client_hash_id"]}  label={"declined" if r["is_declining"] else "held"}  proba={r["forest_proba"]:.3f}')
        print(f'   imp_prev30={r["imp_prev30"]:>9.0f}  clk_prev30={r["clk_prev30"]:>7.0f}  pos={r["pos_prev30"]:>6.1f}  '
              f'days={r["days_with_imp_prev30"]:>2.0f}  age={r["content_age_days"]:>4.0f}d')
    print()

show_cases(flagged, 'Most confident wrong calls (predicted decline, held steady)')
show_cases(missed, 'Biggest misses (declined, model said safe)')

# --- 4) Sensitivity: does the headline lean on fields that are close cousins of the label? ---
# imp_prev30 / clk_prev30 sit next to the label (imp_prev30 is even the label's denominator).
# Like-for-like: the same 200-tree forest, once with all five features, once with the cousins removed.
COUSIN_FREE = ['pos_prev30', 'days_with_imp_prev30', 'content_age_days']
Xc = df[COUSIN_FREE].to_numpy()
sens = {'all': [], 'cousin_free': []}
for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf.fit(X[tr_mask], y[tr_mask])
    tmp = df[te_mask].assign(score=rf.predict_proba(X[te_mask])[:, 1]).sort_values(
        ['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
    sens['all'].append(precision_at_k(tmp['is_declining'], 50))
    rf.fit(Xc[tr_mask], y[tr_mask])
    tmp2 = df[te_mask].assign(score=rf.predict_proba(Xc[te_mask])[:, 1]).sort_values(
        ['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
    sens['cousin_free'].append(precision_at_k(tmp2['is_declining'], 50))

print('Precision@50 mean, all five features (200-tree forest):  %.4f' % (sum(sens['all']) / len(sens['all'])))
print('Precision@50 mean, cousin counts removed (200 trees):    %.4f' % (sum(sens['cousin_free']) / len(sens['cousin_free'])))
print()
print('Reading: the lead shrinks when the click/impression counts leave — the signal is real')
print('but it rests partly on fields that are close relatives of the label. The forest does')
print("not prove that refreshing a page would help; it only ranks pages that look like")
print("today's decliners. Decision-support, not cause.")

Permutation importance — forest (higher = leaned harder; spread = how stable the lean is):
                       fold0   fold1   fold2   fold3   fold4    mean     min     max
days_with_imp_prev30  0.0140  0.0470  0.1943  0.0210  0.0179  0.0588  0.0140  0.1943
content_age_days      0.0168  0.0419  0.1548  0.0113  0.0100  0.0470  0.0100  0.1548
log_clk_prev30        0.0104  0.0164  0.0233  0.0582  0.0220  0.0261  0.0104  0.0582
log_imp_prev30        0.0159  0.0198 -0.0038  0.0357  0.0322  0.0199 -0.0038  0.0357
pos_prev30            0.0076  0.0033 -0.0082  0.0029  0.0018  0.0015 -0.0082  0.0076

Over-flag (proba>=0.7, held steady): 352  |  missed (proba<=0.3, declined): 10,944

--- Most confident wrong calls (predicted decline, held steady) ---
client client_08a6a72ff48e62c0  label=held  proba=0.998
   imp_prev30=      327  clk_prev30=      0  pos=   9.5  days=23  age= 241d
client client_b10cb2997d0c7c86  label=held  proba=0.996
   imp_prev30=      518  clk_prev30=      1  pos=   8.7  d

Precision@50 mean, all five features (200-tree forest):  0.6760
Precision@50 mean, cousin counts removed (200 trees):    0.6520

Reading: the lead shrinks when the click/impression counts leave — the signal is real
but it rests partly on fields that are close relatives of the label. The forest does
not prove that refreshing a page would help; it only ranks pages that look like
today's decliners. Decision-support, not cause.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonym hashes)
- [x] Method chosen from the question (ranking → precision@K): Logistic Regression first, shallow tree, then Random Forest — no boosting, no threshold or hyperparameter tuning on the test fold
- [x] Same rows, same folds, same K, same tie policy as the frozen Week-4 baseline — recompute cross-checked against work/outputs/baseline_folds_receipt.json
- [x] Every client on one side (grouped-by-client split, overlap asserted in code)
- [x] Model vs baseline table with the base rate, all folds kept visible (no single average hides the wobble)
- [x] Errors read: over-flag and miss examples, permutation importance with spread, label-cousin sensitivity
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Receipt written to work/outputs/model_vs_baseline_folds.json
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.